In [1]:
import os
import sys
import pandas as pd
from pathlib import Path
import ast
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
geo_root = repo_root.parent.parent / 'Data' / 'Geospatial'
gus_root = Path(os.getcwd()).parent.parent.parent.parent / "Data" / "GUS"

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

# import local toolkit (try normal import first, fall back to loading from file)
try:
	import inequality_analyzers as inqA
	import local_utility_functions as luf
except Exception:
	import importlib.util
	toolkit_path = repo_root / 'Code' / 'tools' / 'inequality_analyzers.py'
	if toolkit_path.exists():
		spec = importlib.util.spec_from_file_location("inequality_analyzers", str(toolkit_path))
		stk = importlib.util.module_from_spec(spec)
		spec.loader.exec_module(stk)
	else:
		raise


In [2]:
df_demographic = pd.read_csv(gus_root / "data" / 'bdl_demographic_data_OLD.csv', encoding='utf-8')
df_variables = pd.read_csv(gus_root / "metadata" / 'bdl_variables_level6.csv', encoding='utf-8')


In [3]:
def process_subject_data(subjectId):
    """
    Process demographic data for a given subject ID.
    Returns expanded dataframe with flattened structure.
    """
    # Filter by subjectId
    df_subject = df_demographic[df_demographic['subjectId'] == subjectId]
    
    # Get variable metadata for this subject
    variable_ids = df_subject['variableId'].unique()
    df_variables_subset = df_variables[df_variables['id'].isin(variable_ids)][['id', 'n1', 'n2', 'n3', 'n4', 'n5']]
    
    # Merge subject data with variables
    df_merged = pd.merge(df_subject, df_variables_subset, left_on='variableId', right_on='id', how='left')
    
    # Remove constant columns
    for col in ['n1', 'n2', 'n3', 'n4', 'n5']:
        if df_merged[col].nunique() <= 1:
            df_merged = df_merged.drop(columns=[col])
    
    # Parse values column
    def parse_values_column(value):
        try:
            value_list = ast.literal_eval(value)
            if isinstance(value_list, list) and len(value_list) == 1:
                return value_list[0]
            return value_list
        except (ValueError, SyntaxError):
            return None
    
    df_merged['values'] = df_merged['values'].apply(parse_values_column)
    
    # Expand and normalize
    df_expanded = df_merged.explode('values').reset_index(drop=True)
    values_normalized = pd.json_normalize(df_expanded['values'])
    df_expanded = pd.concat([df_expanded.drop(columns=['values']), values_normalized], axis=1)
    
    # Rename and format columns
    df_expanded = df_expanded.rename(columns={"id_x": "nuts_id", "id_y": "var_id"})
    df_expanded['nuts_id'] = df_expanded['nuts_id'].apply(lambda x: str(int(x)).zfill(12))
    df_expanded['teryt_id'] = df_expanded['nuts_id'].apply(luf.nuts_code_to_teryt)
    
    return df_expanded


In [ ]:
df_var = {}
subjects = list(set(df_demographic['subjectId']))
for subject in subjects:
    df_var[subject] = process_subject_data(subject)

KeyboardInterrupt: 